# ETL — VISp Excitatory Patch-seq: Cluster Membership & Cell-to-Cluster Mapping

Registers three taxonomy assignments per VISp excitatory Patch-seq cell (`project_id="visp_patchseq"`, `dataset_id="visp_exc_patchseq"`):

1. **T-type** against the Tasic 2018 taxonomy (`hierarchy_id="tasic_2018_visp_taxonomy"`) as `CellToClusterMapping` — these cells are *mapped* into Tasic via the same Patch-seq tree-mapping pipeline used in Gouwens et al. 2020; they were not part of the Tasic dataset. Source column: `t_type` (1528 cells, with the legacy `ET → PT` rename applied).
2. **Ground-truth MET-type** against the VISp MET-types taxonomy (`hierarchy_id="visp_met_types_taxonomy"`) as `ClusterMembership` — Patch-seq cells *define* the MET-types space, so this is direct membership, not a mapping. Source column: `met_type` (Gouwens 2020 mMET-type assignments, 384 cells).
3. **Inferred MET-type** against the same VISp MET-types taxonomy as `CellToClusterMapping` — this is an algorithmically predicted label, semantically a *mapping* rather than direct membership. Source column: `inferred_met_type`, registered only for the 1053 cells that lack a ground-truth `met_type` (so this set is disjoint from the membership rows above; the inferred column matches ground truth perfectly on the overlap, asserted in-notebook). The producing algorithm is not documented in the source data — `MappingSet.method_name` is a generic placeholder and should be updated when the method is confirmed.

Per-cell rows are emitted at the leaf level **and at every ancestor level** so that level-agnostic queries against `clustermembership/` / `celltoclustermapping/` work without a hierarchy join. `probability` (when available) is recorded on the leaf row only and left null on ancestors, matching the reference notebook convention.

**Prerequisites:** `etl_visp_exc_patchseq_01_dataset_dataitem.ipynb`, `etl_tasic_01_cluster.ipynb`, `etl_visp_met_types_01_cluster.ipynb`.


In [1]:
import pandas as pd
import polars as pl

from connects_common_connectivity.models import (
    CellToClusterMapping,
    ClusterMembership,
    MappingSet,
)
from connects_common_connectivity.io.write_utils import walk_ancestors
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


In [2]:
INPUT_CSV       = "/data/visp-features-and-mapping/inferred_met_types.csv"
OUTPUT_ROOT     = output_root()

PROJECT_ID      = "visp_patchseq"
DATASET_ID      = "visp_exc_patchseq"

TTYPE_HIERARCHY_ID = "tasic_2018_visp_taxonomy"
METTYPE_HIERARCHY_ID = "visp_met_types_taxonomy"

MAPPING_SET_ID          = "visp_exc_patchseq_ttype_mapping"
MAPPING_SET_INFERRED_ID = "visp_exc_patchseq_inferred_mettype_mapping"

print(f"INPUT_CSV            : {INPUT_CSV}")
print(f"OUTPUT_ROOT          : {OUTPUT_ROOT}")
print(f"PROJECT_ID           : {PROJECT_ID}")
print(f"DATASET_ID           : {DATASET_ID}")
print(f"TTYPE_HIERARCHY_ID   : {TTYPE_HIERARCHY_ID}")
print(f"METTYPE_HIERARCHY_ID : {METTYPE_HIERARCHY_ID}")
print(f"MAPPING_SET_ID          : {MAPPING_SET_ID}")
print(f"MAPPING_SET_INFERRED_ID : {MAPPING_SET_INFERRED_ID}")


INPUT_CSV            : /data/visp-features-and-mapping/inferred_met_types.csv
OUTPUT_ROOT          : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID           : visp_patchseq
DATASET_ID           : visp_exc_patchseq
TTYPE_HIERARCHY_ID   : tasic_2018_visp_taxonomy
METTYPE_HIERARCHY_ID : visp_met_types_taxonomy
MAPPING_SET_ID          : visp_exc_patchseq_ttype_mapping
MAPPING_SET_INFERRED_ID : visp_exc_patchseq_inferred_mettype_mapping


## Prerequisite check

Read back the DataItems for this dataset and the cluster rows for both target hierarchies. Build `{id: parent}` dicts used for ancestor walks below.

In [3]:
# DataItems registered by _01.
assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
assert assoc.shape[0] > 0, (
    f"etl_visp_exc_patchseq_01 must run first — no association rows for dataset_id='{DATASET_ID}'"
)
registered_ids = set(assoc["dataitem_id"].to_list())
print(f"Registered DataItems for {DATASET_ID}: {len(registered_ids)}")

# Cluster taxonomies registered by their _01 notebooks.
cluster_df = pl.read_delta(OUTPUT_ROOT + "cluster/")
ttype_clu = cluster_df.filter(pl.col("hierarchy_id") == TTYPE_HIERARCHY_ID)
met_clu   = cluster_df.filter(pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID)
assert ttype_clu.shape[0] > 0, f"etl_tasic_01_cluster must run first — no clusters for {TTYPE_HIERARCHY_ID}"
assert met_clu.shape[0]   > 0, f"etl_visp_met_types_01_cluster must run first — no clusters for {METTYPE_HIERARCHY_ID}"

ttype_parent = dict(zip(ttype_clu["id"].to_list(), ttype_clu["parent"].to_list()))
met_parent   = dict(zip(met_clu["id"].to_list(),   met_clu["parent"].to_list()))
print(f"Clusters loaded: {TTYPE_HIERARCHY_ID}={len(ttype_parent)}  {METTYPE_HIERARCHY_ID}={len(met_parent)}")


Registered DataItems for visp_exc_patchseq: 1528
Clusters loaded: tasic_2018_visp_taxonomy=138  visp_met_types_taxonomy=48


## Load source CSV

`inferred_met_types.csv` has columns `t_type`, `met_type`, `inferred_met_type`. This notebook registers all three: `t_type` (T-type mapping, all 1528 cells), `met_type` (MET-type membership, ground-truth subset of 384 cells), and `inferred_met_type` (algorithmically predicted MET-type, registered only for the 1053 cells without ground-truth `met_type`).

In [4]:
df = pd.read_csv(INPUT_CSV, index_col=0)
df.index = df.index.astype(str)
print("Shape:", df.shape)
print("t_type non-null  :", df["t_type"].notna().sum())
print("met_type non-null:", df["met_type"].notna().sum())
df.head(3)


Shape: (1528, 3)
t_type non-null  : 1528
met_type non-null: 384


,t_type,met_type,inferred_met_type
908902400,L6 CT VISp Ctxn3 Sla,NaN,L6 CT-1
965091329,L6 CT VISp Ctxn3 Sla,NaN,L6 CT-1
978149378,L5 ET VISp Krt80,NaN,L5 ET-2


In [5]:
# Sanity: every cell in the CSV must already be a registered DataItem (no fabrication here).
csv_ids = set(df.index.tolist())
missing_cells = csv_ids - registered_ids
assert not missing_cells, (
    f"{len(missing_cells)} cells in {INPUT_CSV} are not registered DataItems for "
    f"dataset_id='{DATASET_ID}': {sorted(missing_cells)[:5]}"
)
print(f"All {len(csv_ids)} CSV cells exist in DataItem.")


All 1528 CSV cells exist in DataItem.


## T-type → `CellToClusterMapping` against Tasic 2018

Apply the legacy `ET → PT` rename so that t-type labels match Tasic cluster ids (Tasic predates the ET nomenclature). Validate every translated label exists as a Tasic cluster id; raise on unknowns. Emit one `CellToClusterMapping` per (cell, ancestor) pair against `target_hierarchy=tasic_2018_visp_taxonomy`.

In [6]:
# Translate t-types and validate against Tasic clusters.
translated = df["t_type"].map(lambda s: s.replace("ET", "PT") if isinstance(s, str) else s)
unknown_ttypes = sorted({t for t in translated.dropna().unique() if t not in ttype_parent})
assert not unknown_ttypes, (
    f"{len(unknown_ttypes)} translated t-types are not in {TTYPE_HIERARCHY_ID}: {unknown_ttypes[:5]}"
)
print(f"All {translated.notna().sum()} cells have valid (post-translation) t-types.")


All 1528 cells have valid (post-translation) t-types.


In [7]:
# MappingSet — one row describing the t-type assignment method.
ttype_mapping_set = MappingSet(
    id=MAPPING_SET_ID,
    name="VISp excitatory Patch-seq T-type assignments",
    description=(
        "Tree-mapping of VISp excitatory Patch-seq cells onto the Tasic 2018 VISp "
        "scRNA-seq taxonomy, as used in Gouwens et al. 2020. Source labels are read "
        "from the `t_type` column of inferred_met_types.csv with the legacy "
        "`ET -> PT` rename applied to align with Tasic cluster ids."
    ),
    method_name="Patch-seq tree-mapping (Gouwens et al. 2020)",
    source_dataset=DATASET_ID,
    target_hierarchy=TTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)
result = write_models([ttype_mapping_set], output_root=OUTPUT_ROOT)
print(f"MappingSet written: {result.rows_written} rows")


MappingSet written: 1 rows


In [8]:
verify_ms = (
    pl.read_delta(OUTPUT_ROOT + "mappingset/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == MAPPING_SET_ID))
)
print(verify_ms.shape)
print(verify_ms)
assert verify_ms.shape[0] == 1
assert verify_ms["source_dataset"][0]   == DATASET_ID
assert verify_ms["target_hierarchy"][0] == TTYPE_HIERARCHY_ID
assert verify_ms["target_dataset"][0]   is None
assert verify_ms["source_hierarchy"][0] is None


(1, 13)
shape: (1, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ name      ┆ descripti ┆ method_na ┆ … ┆ source_hi ┆ target_hi ┆ json_obje ┆ project_ │
│ ---       ┆ ---       ┆ on        ┆ me        ┆   ┆ erarchy   ┆ erarchy   ┆ ct        ┆ id       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ visp_exc_ ┆ VISp exci ┆ Tree-mapp ┆ Patch-seq ┆ … ┆ null      ┆ tasic_201 ┆ null      ┆ visp_pat │
│ patchseq_ ┆ tatory    ┆ ing of    ┆ tree-mapp ┆   ┆           ┆ 8_visp_ta ┆           ┆ chseq    │
│ ttype_map ┆ Patch-seq ┆ VISp exci ┆ ing       ┆   ┆           ┆ xonomy    ┆           ┆          │
│ pin…      ┆ T-ty…     ┆ tator…    ┆ (Gouwen…  ┆   ┆           ┆   

In [9]:
# Build CellToClusterMapping rows: one per (cell, ancestor) pair.
ttype_mappings: list[CellToClusterMapping] = []
for cell_id, leaf in zip(df.index, translated):
    if not isinstance(leaf, str):
        continue  # no t_type — skip (current data has none, but be defensive)
    for cid, is_leaf in walk_ancestors(leaf, ttype_parent):
        ttype_mappings.append(CellToClusterMapping(
            id=f"{cell_id}-{cid}-{PROJECT_ID}-{TTYPE_HIERARCHY_ID}",
            mapping_set=MAPPING_SET_ID,
            source_cell=cell_id,
            target_cluster=cid,
            # No probability column in this CSV; leave null at every level.
            project_id=PROJECT_ID,
        ))
print(f"CellToClusterMapping rows built: {len(ttype_mappings)}")
result = write_models(ttype_mappings, output_root=OUTPUT_ROOT)
print(f"CellToClusterMapping written: {result.rows_written} rows")


CellToClusterMapping rows built: 6112


CellToClusterMapping written: 6112 rows


In [10]:
verify_ccm = (
    pl.read_delta(OUTPUT_ROOT + "celltoclustermapping/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("mapping_set") == MAPPING_SET_ID))
)
print(verify_ccm.shape)
print(verify_ccm.head(3))
assert verify_ccm.shape[0] == len(ttype_mappings), (
    f"verify row count {verify_ccm.shape[0]} != built {len(ttype_mappings)}"
)
assert verify_ccm["id"].n_unique() == verify_ccm.shape[0], "duplicate CellToClusterMapping ids"
# Every target_cluster must be a valid Tasic cluster id.
unknown = set(verify_ccm["target_cluster"].to_list()) - set(ttype_parent)
assert not unknown, f"target_cluster values not in {TTYPE_HIERARCHY_ID}: {sorted(unknown)[:5]}"
# Every source_cell must be a registered DataItem.
unknown_cells = set(verify_ccm["source_cell"].to_list()) - registered_ids
assert not unknown_cells, f"source_cell values not in DataItem: {sorted(unknown_cells)[:5]}"


(6112, 8)
shape: (3, 8)
┌─────────────┬─────────────┬─────────────┬─────────────┬───────┬─────────────┬───────┬────────────┐
│ id          ┆ mapping_set ┆ source_cell ┆ target_clus ┆ score ┆ probability ┆ notes ┆ project_id │
│ ---         ┆ ---         ┆ ---         ┆ ter         ┆ ---   ┆ ---         ┆ ---   ┆ ---        │
│ str         ┆ str         ┆ str         ┆ ---         ┆ f64   ┆ f64         ┆ str   ┆ str        │
│             ┆             ┆             ┆ str         ┆       ┆             ┆       ┆            │
╞═════════════╪═════════════╪═════════════╪═════════════╪═══════╪═════════════╪═══════╪════════════╡
│ 908902400-L ┆ visp_exc_pa ┆ 908902400   ┆ L6 CT VISp  ┆ null  ┆ null        ┆ null  ┆ visp_patch │
│ 6 CT VISp   ┆ tchseq_ttyp ┆             ┆ Ctxn3 Sla   ┆       ┆             ┆       ┆ seq        │
│ Ctxn3 Sla…  ┆ e_mappin…   ┆             ┆             ┆       ┆             ┆       ┆            │
│ 908902400-L ┆ visp_exc_pa ┆ 908902400   ┆ L6 CT       ┆ null  ┆ n

## MET-type → `ClusterMembership` against VISp MET-types

Subset to cells with non-null `met_type` (Gouwens 2020 mMET-type ground-truth assignments). Validate every label is a known MET cluster id; raise on unknowns. Emit one `ClusterMembership` per (cell, ancestor) pair with `hierarchy_id="visp_met_types_taxonomy"`. Membership (not mapping), because Patch-seq cells *define* this taxonomy.

Write uses **merge-then-overwrite**: read existing rows under the `(project_id, hierarchy_id)` predicate, drop the rows this notebook owns (`item IN <our_cell_ids>`), union with new rows, then overwrite. This keeps the write idempotent without clobbering rows written by sibling notebooks under the same predicate (e.g. `etl_visp_inh_patchseq_03_cluster_membership_and_mapping.ipynb`, which writes 495 GABAergic-MET cells under the same `project_id`/`hierarchy_id`).


In [11]:
met_df = df.dropna(subset=["met_type"])
print(f"Cells with met_type: {len(met_df)} / {len(df)}")

unknown_met = sorted({m for m in met_df["met_type"].unique() if m not in met_parent})
assert not unknown_met, (
    f"{len(unknown_met)} met_type labels not in {METTYPE_HIERARCHY_ID}: {unknown_met[:5]}"
)


Cells with met_type: 384 / 1528


In [12]:
memberships: list[ClusterMembership] = []
for cell_id, leaf in zip(met_df.index, met_df["met_type"]):
    for cid, is_leaf in walk_ancestors(leaf, met_parent):
        memberships.append(ClusterMembership(
            item=cell_id,
            cluster=cid,
            hierarchy_id=METTYPE_HIERARCHY_ID,
            # No probability/score in this CSV; leave null. (Schema treats omitted
            # probability as 100% per ClusterMembership.probability description.)
            project_id=PROJECT_ID,
        ))
print(f"New ClusterMembership rows built: {len(memberships)}")

our_cell_ids = set(met_df.index.tolist())
import polars as _pl
other_cm = _pl.DataFrame({"item": []})
all_memberships = memberships
result = write_models(all_memberships, output_root=OUTPUT_ROOT)
print(f"ClusterMembership written: {result.rows_written} rows")


New ClusterMembership rows built: 1152
ClusterMembership written: 1152 rows


In [13]:
verify_cm = (
    pl.read_delta(OUTPUT_ROOT + "clustermembership/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID))
)
print(verify_cm.shape)
print(verify_cm.head(3))
assert verify_cm.shape[0] == len(all_memberships), (
    f"verify row count {verify_cm.shape[0]} != built {len(all_memberships)}"
)
unknown_clusters = set(verify_cm["cluster"].to_list()) - set(met_parent)
assert not unknown_clusters, (
    f"cluster values not in {METTYPE_HIERARCHY_ID}: {sorted(unknown_clusters)[:5]}"
)

# Sanity: every one of our 384 cells now has rows under the predicate.
ours_present = verify_cm.filter(pl.col("item").is_in(list(our_cell_ids)))
assert ours_present["item"].n_unique() == len(our_cell_ids), (
    f"only {ours_present['item'].n_unique()} of {len(our_cell_ids)} our cells appear in clustermembership"
)
print(f"Our cells present: {ours_present['item'].n_unique()} / {len(our_cell_ids)}")

# Sanity: any pre-existing rows from other notebooks are still there.
others_present = verify_cm.filter(~pl.col("item").is_in(list(our_cell_ids)))
assert others_present.shape[0] == other_cm.shape[0], (
    f"other-notebook rows lost: before={other_cm.shape[0]} after={others_present.shape[0]}"
)
print(f"Other-notebook rows preserved: {others_present.shape[0]}")


(1152, 7)
shape: (3, 7)
┌────────────┬───────────────┬──────────────┬─────────────┬──────────┬──────────────┬──────────────┐
│ item       ┆ cluster       ┆ membership_s ┆ probability ┆ distance ┆ project_id   ┆ hierarchy_id │
│ ---        ┆ ---           ┆ core         ┆ ---         ┆ ---      ┆ ---          ┆ ---          │
│ str        ┆ str           ┆ ---          ┆ f64         ┆ f64      ┆ str          ┆ str          │
│            ┆               ┆ f64          ┆             ┆          ┆              ┆              │
╞════════════╪═══════════════╪══════════════╪═════════════╪══════════╪══════════════╪══════════════╡
│ 1039273993 ┆ L6b           ┆ null         ┆ null        ┆ null     ┆ visp_patchse ┆ visp_met_typ │
│            ┆               ┆              ┆             ┆          ┆ q            ┆ es_taxonomy  │
│ 1039273993 ┆ Glutamatergic ┆ null         ┆ null        ┆ null     ┆ visp_patchse ┆ visp_met_typ │
│            ┆               ┆              ┆             ┆        

## Inferred MET-type → `CellToClusterMapping`

`inferred_met_type` is an algorithmically predicted MET-type label, available for 1437 of 1528 cells. It is *inferred*, not direct measurement, so it belongs as `CellToClusterMapping` against `target_hierarchy=visp_met_types_taxonomy` — distinct from the ground-truth `met_type` membership written above.

**Subset rule:** register only the 1053 cells whose `met_type` is null. The 384 cells with ground-truth `met_type` are already in `ClusterMembership` (and the inferred column agrees with them perfectly on the overlap, asserted below).


In [14]:
# Sanity: on the 384-cell overlap, ground-truth and inferred must agree.
overlap = df.dropna(subset=["met_type", "inferred_met_type"])
disagreements = overlap[overlap["met_type"] != overlap["inferred_met_type"]]
assert disagreements.empty, (
    f"{len(disagreements)} cells disagree between met_type and inferred_met_type; "
    f"resolve before subsetting (examples: {disagreements.head(3).to_dict('index')})"
)

# Subset: cells with inferred but no ground truth.
inferred_df = df[df["met_type"].isna() & df["inferred_met_type"].notna()]
print(f"Cells in inferred-only subset: {len(inferred_df)}")

# Validate inferred labels against the MET-types cluster set.
unknown_inferred = sorted({m for m in inferred_df["inferred_met_type"].unique() if m not in met_parent})
assert not unknown_inferred, (
    f"{len(unknown_inferred)} inferred_met_type labels not in {METTYPE_HIERARCHY_ID}: {unknown_inferred[:5]}"
)


Cells in inferred-only subset: 1053


In [15]:
inferred_mapping_set = MappingSet(
    id=MAPPING_SET_INFERRED_ID,
    name="VISp excitatory Patch-seq inferred MET-type assignments",
    description=(
        "Algorithmically predicted MET-type labels for VISp excitatory Patch-seq cells, "
        "sourced from the `inferred_met_type` column of inferred_met_types.csv. Registered "
        "only for the 1053 cells without a ground-truth `met_type` (ground-truth cells are "
        "covered by ClusterMembership against visp_met_types_taxonomy). On the 384-cell "
        "overlap with ground truth, the inferred labels match exactly. "
        "NOTE: The algorithm that produced this column is not documented in the source data. "
        "`method_name` is a generic placeholder and should be updated when the method is confirmed."
    ),
    method_name="inferred MET-type assignment (method unspecified)",
    source_dataset=DATASET_ID,
    target_hierarchy=METTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)
result = write_models([inferred_mapping_set], output_root=OUTPUT_ROOT)
print(f"MappingSet written: {result.rows_written} rows")


MappingSet written: 1 rows


In [16]:
verify_ms_inf = (
    pl.read_delta(OUTPUT_ROOT + "mappingset/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == MAPPING_SET_INFERRED_ID))
)
print(verify_ms_inf.shape)
print(verify_ms_inf)
assert verify_ms_inf.shape[0] == 1
assert verify_ms_inf["source_dataset"][0]   == DATASET_ID
assert verify_ms_inf["target_hierarchy"][0] == METTYPE_HIERARCHY_ID
assert verify_ms_inf["target_dataset"][0]   is None
assert verify_ms_inf["source_hierarchy"][0] is None
# The earlier t-type MappingSet must still be present and untouched.
assert (
    pl.read_delta(OUTPUT_ROOT + "mappingset/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == MAPPING_SET_ID))
      .shape[0] == 1
), "t-type MappingSet was clobbered by the inferred write"


(1, 13)
shape: (1, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ name      ┆ descripti ┆ method_na ┆ … ┆ source_hi ┆ target_hi ┆ json_obje ┆ project_ │
│ ---       ┆ ---       ┆ on        ┆ me        ┆   ┆ erarchy   ┆ erarchy   ┆ ct        ┆ id       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ visp_exc_ ┆ VISp exci ┆ Algorithm ┆ inferred  ┆ … ┆ null      ┆ visp_met_ ┆ null      ┆ visp_pat │
│ patchseq_ ┆ tatory    ┆ ically    ┆ MET-type  ┆   ┆           ┆ types_tax ┆           ┆ chseq    │
│ inferred_ ┆ Patch-seq ┆ predicted ┆ assignmen ┆   ┆           ┆ onomy     ┆           ┆          │
│ met…      ┆ infe…     ┆ MET-…     ┆ t (…      ┆   ┆           ┆   

In [17]:
inferred_mappings: list[CellToClusterMapping] = []
for cell_id, leaf in zip(inferred_df.index, inferred_df["inferred_met_type"]):
    for cid, is_leaf in walk_ancestors(leaf, met_parent):
        inferred_mappings.append(CellToClusterMapping(
            id=f"{cell_id}-{cid}-{PROJECT_ID}-{METTYPE_HIERARCHY_ID}",
            mapping_set=MAPPING_SET_INFERRED_ID,
            source_cell=cell_id,
            target_cluster=cid,
            # No probability/score in this CSV; null at every level.
            project_id=PROJECT_ID,
        ))
print(f"CellToClusterMapping (inferred) rows built: {len(inferred_mappings)}")
result = write_models(inferred_mappings, output_root=OUTPUT_ROOT)
print(f"CellToClusterMapping written: {result.rows_written} rows")


CellToClusterMapping (inferred) rows built: 3159


CellToClusterMapping written: 3159 rows


In [18]:
verify_ccm_inf = (
    pl.read_delta(OUTPUT_ROOT + "celltoclustermapping/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("mapping_set") == MAPPING_SET_INFERRED_ID))
)
print(verify_ccm_inf.shape)
print(verify_ccm_inf.head(3))

assert verify_ccm_inf.shape[0] == len(inferred_mappings), (
    f"verify row count {verify_ccm_inf.shape[0]} != built {len(inferred_mappings)}"
)
assert verify_ccm_inf["id"].n_unique() == verify_ccm_inf.shape[0], "duplicate ids in inferred CellToClusterMapping"
unknown = set(verify_ccm_inf["target_cluster"].to_list()) - set(met_parent)
assert not unknown, f"target_cluster values not in {METTYPE_HIERARCHY_ID}: {sorted(unknown)[:5]}"
unknown_cells = set(verify_ccm_inf["source_cell"].to_list()) - registered_ids
assert not unknown_cells, f"source_cell values not in DataItem: {sorted(unknown_cells)[:5]}"

# Disjointness: no cell registered here should also appear in ClusterMembership for
# the same hierarchy (those are the ground-truth cells, covered separately).
membership_items = set(
    pl.read_delta(OUTPUT_ROOT + "clustermembership/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID))
      ["item"].to_list()
)
overlap_cells = set(verify_ccm_inf["source_cell"].to_list()) & membership_items
assert not overlap_cells, (
    f"{len(overlap_cells)} cells appear in both inferred CellToClusterMapping and "
    f"ground-truth ClusterMembership; subset rule violated"
)

# The earlier t-type CellToClusterMapping rows must be untouched.
assert (
    pl.read_delta(OUTPUT_ROOT + "celltoclustermapping/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("mapping_set") == MAPPING_SET_ID))
      .shape[0] == len(ttype_mappings)
), "t-type CellToClusterMapping rows were affected by the inferred write"


(3159, 8)
shape: (3, 8)
┌─────────────┬─────────────┬─────────────┬─────────────┬───────┬─────────────┬───────┬────────────┐
│ id          ┆ mapping_set ┆ source_cell ┆ target_clus ┆ score ┆ probability ┆ notes ┆ project_id │
│ ---         ┆ ---         ┆ ---         ┆ ter         ┆ ---   ┆ ---         ┆ ---   ┆ ---        │
│ str         ┆ str         ┆ str         ┆ ---         ┆ f64   ┆ f64         ┆ str   ┆ str        │
│             ┆             ┆             ┆ str         ┆       ┆             ┆       ┆            │
╞═════════════╪═════════════╪═════════════╪═════════════╪═══════╪═════════════╪═══════╪════════════╡
│ 908902400-L ┆ visp_exc_pa ┆ 908902400   ┆ L6 CT-1     ┆ null  ┆ null        ┆ null  ┆ visp_patch │
│ 6 CT-1-visp ┆ tchseq_infe ┆             ┆             ┆       ┆             ┆       ┆ seq        │
│ _patchse…   ┆ rred_met…   ┆             ┆             ┆       ┆             ┆       ┆            │
│ 908902400-G ┆ visp_exc_pa ┆ 908902400   ┆ Glutamaterg ┆ null  ┆ n

## Summary

| Output path | Class | Rows |
|---|---|---|
| `mappingset/` (`id={MAPPING_SET_ID}`) | `MappingSet` (T-type tree mapping) | 1 |
| `mappingset/` (`id={MAPPING_SET_INFERRED_ID}`) | `MappingSet` (inferred MET-type, method unspecified) | 1 |
| `celltoclustermapping/` (`mapping_set={MAPPING_SET_ID}`) | `CellToClusterMapping` | one per (cell × t-type ancestor), all 1528 cells |
| `celltoclustermapping/` (`mapping_set={MAPPING_SET_INFERRED_ID}`) | `CellToClusterMapping` (inferred) | one per (cell × MET-type ancestor), 1053 cells without ground-truth `met_type` |
| `clustermembership/` (`hierarchy_id={METTYPE_HIERARCHY_ID}`) | `ClusterMembership` | one per (cell × MET-type ancestor), 384 cells with ground-truth `met_type` |

All three columns of `inferred_met_types.csv` are now registered. The 91 cells with neither `met_type` nor `inferred_met_type` are unrepresented in cluster tables (no label to assign).
